In [1]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from pathlib import Path
import torch

from utils.dataset import cycle, ImageEditDataset
import torchvision.transforms.functional as TF


In [2]:
DATASET_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset_v1.0.0/metadata_edit.csv"
)
meta_df = pd.read_csv(DATASET_PATH)

In [3]:
dataset = ImageEditDataset(DATASET_PATH)


dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=1, shuffle=True, num_workers=8
)

dataloader = cycle(dataloader)

In [144]:
meta_csv = pd.read_csv(DATASET_PATH)

In [ ]:
import shutil

In [ ]:
DISTILL_DATASET = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset"
)

DISTILL_DATASET_V1 = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset_v1.0.0"
)

DISTILL_DATASET_V1.mkdir(exist_ok=True, parents=True)


META_CSV_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset/metadata_edit.csv"
)

new_meta_df = pd.DataFrame(columns=["image", "edit_image", "prompt"])

meta_df = pd.read_csv(META_CSV_PATH)


for idx, row in meta_df.iterrows():
    prompt = row["prompt"]
    image = row["image"]
    edit_image = row["edit_image"]

    if prompt == "empty room":
        new_img_path = DISTILL_DATASET_V1 / "empty_room" / image
        new_edit_img_path = DISTILL_DATASET_V1 / "empty_room" / edit_image

        new_img = f"empty_room/{image}"
        new_edit_img = f"empty_room/{edit_image}"

    else:
        new_img_path = DISTILL_DATASET_V1 / "clutter_removal" / image
        new_edit_img_path = DISTILL_DATASET_V1 / "clutter_removal" / edit_image

        new_img = f"clutter_removal/{image}"
        new_edit_img = f"clutter_removal/{edit_image}"

    ori_img_path = DISTILL_DATASET / image
    ori_edit_img_path = DISTILL_DATASET / edit_image

    new_meta_df.loc[idx] = [
        new_img,
        new_edit_img,
        prompt,
    ]
    new_img_path.parent.mkdir(exist_ok=True, parents=True)
    new_edit_img_path.parent.mkdir(exist_ok=True, parents=True)
    shutil.copyfile(ori_img_path, new_img_path)
    shutil.copyfile(ori_edit_img_path, new_edit_img_path)

new_meta_df.to_csv(DISTILL_DATASET_V1 / "metadata_edit.csv", index=False)


In [ ]:
clutter_removal_1 = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_image_edit_dataset_v3.0.1/clutter_to_furnished/1xgw"
)

clutter_removal_2 = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_image_edit_dataset_v3.0.1/clutter_to_furnished/19pL"
)

clutter_removal_3 = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_image_edit_dataset_v3.0.1/clutter_to_furnished/neo"
)

In [ ]:
clutter_removal_img_paths = list(clutter_removal_2.glob("*_edit.jpg"))


In [ ]:
import random

selected_paths = random.sample(clutter_removal_img_paths, 734)


In [ ]:
len(selected_paths)


In [ ]:
DATASET_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset_v1.0.0/clutter_removal"
)

META_CSV_PATH = Path(
    "/home/rick-mbp/Diffusion-Distillation/qwen_training_data/qwen_distill_dataset_v1.0.0/metadata_edit.csv"
)

meta_csv = pd.read_csv(META_CSV_PATH)


for clutter_removal_img_path in selected_paths:
    img_name = clutter_removal_img_path.name
    target_img_name = img_name.replace("_edit", "")
    target_img_path = clutter_removal_img_path.parent / target_img_name
    folder = clutter_removal_img_path.parent.name

    copy_edit_img_path = DATASET_PATH / folder / img_name
    copy_img_path = DATASET_PATH / folder / target_img_name

    copy_edit_img_path.parent.mkdir(exist_ok=True, parents=True)
    copy_img_path.parent.mkdir(exist_ok=True, parents=True)

    shutil.copyfile(clutter_removal_img_path, copy_edit_img_path)
    shutil.copyfile(target_img_path, copy_img_path)

    new_meta_line = [
        f"clutter_removal/{folder}/{target_img_name}",
        f"clutter_removal/{folder}/{img_name}",
        "clean room",
    ]
    meta_csv.loc[len(meta_csv)] = new_meta_line

meta_csv.to_csv(META_CSV_PATH, index=False)

In [ ]:
num_empty_room = meta_csv["prompt"] == "empty room"
num_clutter_removal = meta_csv["prompt"] == "clean room"

print(num_empty_room.sum())
print(num_clutter_removal.sum())

LACK_CLEAN_ROOM_NUM = num_empty_room.sum() - num_clutter_removal.sum()

print(LACK_CLEAN_ROOM_NUM)
